# Minimal working example using BERT (`distBERT` model)

In [48]:
# !pip install pandas torch transformers faiss-cpu
!pip install faiss-cpu

## Setup

In [49]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
import faiss

## Pseudo data creation

In [50]:
# ----------------------------
# 1) Sample pseudo "data"
# ----------------------------
# items = pd.DataFrame([
#     {"item_id":"i1","title":"Noise Cancelling Headphones","description":"Wireless noise-cancelling headphones with 30-hour battery life","category":"electronics"},
#     {"item_id":"i2","title":"Mechanical Keyboard","description":"Mechanical keyboard with RGB and hot-swappable switches","category":"electronics"},
#     {"item_id":"i3","title":"Running Shoes","description":"Running shoes designed for long distance comfort and stability","category":"sports"},
#     {"item_id":"i4","title":"Vegetarian Cookbook","description":"Cookbook featuring quick vegetarian recipes for busy weeknights","category":"books"},
#     {"item_id":"i5","title":"Fitness Smartwatch","description":"Smartwatch with heart rate monitoring, GPS, and sleep tracking","category":"electronics"},
# ])

# events = pd.DataFrame([
#     {"user_id":"u1","item_id":"i1","ts":"2026-01-10"},
#     {"user_id":"u1","item_id":"i5","ts":"2026-01-11"},
#     {"user_id":"u2","item_id":"i2","ts":"2026-01-12"},
#     {"user_id":"u3","item_id":"i3","ts":"2026-01-13"},
#     {"user_id":"u3","item_id":"i4","ts":"2026-01-14"},
# ])

items = pd.read_csv("/content/items.csv")
events = pd.read_csv("/content/events.csv")

events["ts"] = pd.to_datetime(events["ts"])

print(items.columns)
print(events.columns)
items.head()
events.dtypes

# set of pseudo data (items), all are dictionaries
# has just 2 columns: title and description
# events - did they look at this item, browse this item, at the date (ts = timestamp)
# events["ts"] line converts string data into a real date

Index(['item_id', 'title', 'description', 'category', 'brand', 'price',
       'tags'],
      dtype='object')
Index(['user_id', 'item_id', 'ts', 'event_type', 'session_id', 'device',
       'dwell_seconds', 'price_at_event', 'category_at_event'],
      dtype='object')


,0
user_id,object
item_id,object
ts,datetime64[ns]
event_type,object
session_id,object
device,object
dwell_seconds,int64
price_at_event,float64
category_at_event,object


## BERT encoder (What makes BERT work?)

In [51]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# architectural stuff

MODEL_NAME = "distilbert-base-uncased"  # for illustration, 66M model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME) # converts raw text --> token ids/ numbers that model can understand
encoder = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE) # actual model
# how to call model
# uncased - means capitalizaiton doesn't matter
# distilled bert - student model of bert (parameters are smaller)
# encoder - comes from model - uses gpu (to(DEVICE))
# tokenizer - comes from tokenzier

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


**Evaluate the BERT encoder**

In [52]:
# ----------------------------
# 2) BERT encode helper
# ----------------------------

# turn off gradient calculation (no back propagation in using BERT)
@torch.no_grad()
def bert_embed(texts, max_len=256): # returns text/string into numbers/tokens
    batch = tokenizer(
        texts, padding=True, truncation=True, max_length=max_len, return_tensors="pt"
    ) # conversion
    batch = {k: v.to(DEVICE) for k, v in batch.items()}
    out = encoder(**batch)
    cls = out.last_hidden_state[:, 0]          # first column [CLS]-like token for classification
    emb = F.normalize(cls, dim=-1)             # normalization
    return emb.cpu().numpy().astype("float32") # size (B, 768)

# want to go from text to tokens ("embedding", first step of tokenizer)
# set max_len - how long should the vector output be
    # as u increase embedding size, you try to capture more info
# texts - exact block of text you are working with
# batch - put v (values).toDEVICE() because we want the values to encode

## Embedding items into vectors (for comparisons)

In [53]:
# ----------------------------
# 3) Offline job: item embeddings
# ----------------------------

# items["text"] = items["title"] + ". " + items["description"]

def build_item_text(i):
  features = {
      str(i["title"]),
      str(i["description"]),
      f"Category: {i['category']}",
      f"Brand: {i['brand']}",
      f"Price: ${i['price']}",
      f"Tags: {i['tags']}",
  }
  features = [f for f in features if "nan" not in f.lower()]
  return ". ".join(features)

items["text"] = items.apply(build_item_text, axis=1)
item_vecs = bert_embed(items["text"].tolist())
item_id_list = items["item_id"].tolist()
# want title + description in one text
# print out some of these items before moving on to make sure you know what they're doing

# Build ANN index (inner product works with normalized vectors)
index = faiss.IndexFlatIP(item_vecs.shape[1])
index.add(item_vecs)
# way to compare vectors
# builds similarity comparison

## What user-specific data are there?

In [54]:
# ----------------------------
# 4) Feature builder: user text from last N clicks
# ----------------------------
def build_user_text(user_id, events, items, N=10): # builds text representation of user based on their history
    hist = (events[events["user_id"] == user_id]
            .sort_values("ts")
            .tail(N)["item_id"]
            .tolist())
    if not hist:
        return "no history"
    text = items.set_index("item_id").loc[hist, "text"].tolist()
    return " ".join(text), set(hist)

# what to recommend to user based on items
# if they have looked at 5 different things, only grab most recent 3 (n) things they saw = their history
# then find/form the text

## How to recommend "similar" item with BERT?

In [55]:
# ----------------------------
# 5) "What to recommend" function
# ----------------------------
def recommend(user_id, k=5): # embeds user history into the same size as items, then finds the most similar item
    user_text, seen = build_user_text(user_id, events, items, N=10)
    u = bert_embed([user_text])  # (1, 768)
    scores, idx = index.search(u, k + len(seen))  # keep track of what was seen by the user
    recs = []
    for j in idx[0]:
        iid = item_id_list[j]
        if iid not in seen:
            recs.append(iid)
        if len(recs) == k:
            break
    return recs # returns k recommendations

# recommending 3 things
# build user text for user, embed the user text, look for similarities to all items we have,

In [56]:

# for u in ["u1","u2","u3"]:
#     print(u, "->", recommend(u, k=3))

for u in ["u1", "u2", "u3"]:
    user_events = events[events["user_id"] == u].sort_values("ts")
    print("\nUser Events: ", user_events)

    recs = recommend(u, k=3)
    print("\nRecommended Item IDs: ", recs)

    rec_items = items[items["item_id"].isin(recs)]
    print("\nRecommended Items: ", rec_items)
# three users and their top 3 recommendations


User Events:     user_id item_id                  ts   event_type       session_id   device  \
0       u1      i2 2026-01-01 07:47:00        click  s_u1_20260101_1   tablet   
1       u1     i13 2026-01-04 08:01:00         view  s_u1_20260104_2   tablet   
2       u1      i2 2026-01-07 14:28:00        click  s_u1_20260107_3   mobile   
3       u1     i13 2026-01-07 19:06:00         view  s_u1_20260107_1   tablet   
4       u1      i1 2026-01-10 08:42:00  add_to_cart  s_u1_20260110_1  desktop   
5       u1      i7 2026-01-10 10:59:00        click  s_u1_20260110_2   tablet   
6       u1      i6 2026-01-12 21:40:00         view  s_u1_20260112_1   tablet   
7       u1     i17 2026-01-13 14:10:00        click  s_u1_20260113_2   tablet   
8       u1     i11 2026-01-15 08:14:00         view  s_u1_20260115_3   mobile   
9       u1     i15 2026-01-18 19:56:00         view  s_u1_20260118_2   mobile   
10      u1      i2 2026-01-20 20:57:00     purchase  s_u1_20260120_1   mobile   
11      u1   

In [57]:
# high-level, in the functions we have to change the columns that get passed into the encoder
# 3) offline embedding - do more here than just description
# max_len - make larger
# 4) building user text -